# Урок 24. Этика ИИ: смещение данных и ответственность

**Дата:** 04.03.2027 · **60 минут** · 8–9 класс  
**Модуль 4.** Машинное обучение (уроки 17–25)  
**Опирается на:** урок 17 (признаки, утечка), уроки 19–21 (модели, метрики, ошибки), урок 12 (скрытые факторы).

Модели всё чаще влияют на людей: кого пригласить на собеседование, кому выдать кредит, кого взять в команду, какие новости показать. Ошибка модели здесь — не просто «неверная строка в таблице», а чья-то упущенная возможность. Сегодня увидим, как модель с хорошей общей accuracy оказывается **несправедливой** к целой группе людей, и научимся это замечать.

## Цели урока

К концу занятия ты сможешь:

1. Называть главные принципы ответственного ИИ.
2. Объяснять, откуда в моделях берётся **смещение** (bias).
3. Проверять качество модели **по группам**, а не только в среднем.
4. Соблюдать правила работы с **персональными данными**.
5. Заполнять **карточку ограничений** модели.

## План на 60 минут

| Минуты | Что делаем |
|-------:|------------|
| 0–10 | Кейсы: когда ИИ вредит |
| 10–20 | Откуда берётся смещение |
| 20–35 | Эксперимент: метрики по группам |
| 35–42 | Приватность и ответственность |
| 42–55 | Практика: разбор ситуаций, карточка ограничений |
| 55–60 | Итог |

## 1. Принципы ответственного ИИ

| Принцип | Вопрос инженера |
|---------|-----------------|
| **Справедливость** | Одинаково ли хорошо модель работает для разных групп людей? |
| **Приватность** | Есть ли согласие людей? Не утекут ли их данные? |
| **Прозрачность** | Можем ли объяснить, почему модель так решила? |
| **Ответственность** | Кто отвечает за ошибку? Может ли человек оспорить решение? |
| **Безопасность** | Что будет при ошибке? Есть ли человек, который проверит? |

Это не «добавка в конце проекта», а вопросы, которые задают на **каждом** шаге цикла: при постановке задачи, сборе данных, выборе метрики, демонстрации.

## 2. Кейсы для обсуждения (5 минут)

Обсудите в парах: кому и как мог навредить каждый случай?

1. **Отбор резюме.** Модель обучили на резюме сотрудников, которых компания нанимала 10 лет. Раньше в компании почти не было женщин-инженеров. Модель начала занижать оценку резюме, где упоминаются «женские» кружки и школы.
2. **Распознавание лиц.** Система хорошо узнаёт людей со светлой кожей и заметно хуже — с тёмной: в обучающих фото их было мало.
3. **Рекомендации видео.** Система показывает то, что смотрят дольше всего. Подросток, который однажды посмотрел видео о диетах, получает всё более экстремальный контент.
4. **Проверка сочинений.** Модель ставит низкие баллы необычному стилю письма, потому что училась на «шаблонных» сочинениях.

Подобные истории случались с реальными системами — именно после них в индустрии стали проверять модели на справедливость.

## 3. Откуда берётся смещение

**Смещение** (*bias*) — систематическая ошибка, из-за которой модель хуже работает для части людей или повторяет несправедливость. Модель не «злая»: она честно учится на том, что ей дали.

| Источник | Что происходит | Пример |
|----------|----------------|--------|
| **Историческое смещение** | в разметке записаны несправедливые решения прошлого | старые решения о найме → модель их копирует |
| **Нерепрезентативные данные** | одной группы в данных мало | мало фото людей с тёмной кожей |
| **Признак-посредник** | признак косвенно указывает на группу | район проживания связан с доходом семьи |
| **Неравное измерение** | признак значит разное для разных групп | посещаемость у очных и онлайн-учеников |
| **Неудачная цель** | модель оптимизирует не то, что важно людям | «время просмотра» вместо пользы |

## 4. Эксперимент: справедлива ли модель?

**Ситуация.** Кружок набирает учеников на продвинутый курс и хочет взять тех, кто, по прогнозу модели, успешно сдаст итоговую работу. Признаки: посещаемость **очных** занятий и число сданных домашних заданий.

В кружке есть и **онлайн-ученики** — их около 12%. Они занимаются из дома, поэтому их посещаемость очных занятий почти нулевая, хотя учатся они не хуже. Модели об этом никто не сказал.

In [ ]:
import random
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, recall_score

def make_club(n, seed):
    """Ученики очного и онлайн-формата. «Старание» скрыто и определяет успех."""
    random.seed(seed)
    rows = []
    for i in range(n):
        online = random.random() < 0.12
        effort = random.random()                         # скрытое «старание» от 0 до 1
        if online:
            form = "онлайн"
            attendance = random.randint(10, 40)          # на очные занятия почти не ходят
        else:
            form = "очно"
            attendance = min(100, max(20, round(30 + 70 * effort + random.gauss(0, 8))))
        homework = min(10, max(0, round(10 * effort + random.gauss(0, 2.5))))
        rows.append({
            "формат": form,
            "посещаемость": attendance,
            "сдано_дз": homework,
            "сдал": effort + random.gauss(0, 0.1) > 0.45,
        })
    return pd.DataFrame(rows)

df = make_club(500, seed=26)
df.groupby("формат")["сдал"].agg(["mean", "count"]).round(2)

Онлайн-ученики сдают не хуже очных. Обучим модель **без** столбца «формат» — кажется, что так честнее: модель «не знает», кто откуда.

In [ ]:
X = df[["посещаемость", "сдано_дз"]]
y = df["сдал"]
groups = df["формат"]

X_train, X_test, y_train, y_test, g_train, g_test = train_test_split(
    X, y, groups, test_size=0.3, random_state=42, stratify=y
)

model = DecisionTreeClassifier(max_depth=3, random_state=0)
model.fit(X_train, y_train)
pred = model.predict(X_test)
print("Общая accuracy на test:", round(accuracy_score(y_test, pred), 2))

`train_test_split` умеет делить сразу несколько столбцов согласованно: мы передали ещё и `groups`, чтобы знать, к какой группе относится каждый ученик test.

Общая accuracy выглядит неплохо. Теперь главное — **посчитаем метрики отдельно для каждой группы**:

In [ ]:
for group in ["очно", "онлайн"]:
    mask = (g_test == group).to_numpy()
    acc = accuracy_score(y_test[mask], pred[mask])
    rec = recall_score(y_test[mask], pred[mask])
    print(f"{group:7}: учеников в test {mask.sum():3}, accuracy {acc:.2f}, recall {rec:.2f}")

`.to_numpy()` превращает столбец в массив `True / False`. Дальше `pred[mask]` и `y_test[mask]` работают как фильтр из урока 10: оставляют только тех учеников, у которых в маске `True`, то есть только одну группу.

**Что произошло.** Для онлайн-учеников модель ошибается в большинстве случаев и «не видит» две трети тех, кто справился бы (recall около 0.33). Модель выучила правило «низкая посещаемость → не сдаст», которое верно для очников, но **ложно** для онлайн-учеников. В среднем по всем ученикам этого не видно: онлайн-учеников мало, и их ошибки «тонут» в общей accuracy.

Мы убрали «формат» из признаков, но несправедливость осталась: **посещаемость** оказалась признаком-посредником, который значит разное для разных групп.

### Как исправить в этом случае

Здесь формат обучения — законная и важная информация: он объясняет, **как измерялась** посещаемость. Добавим его в признаки (в виде `True / False`) и снова проверим по группам.

In [ ]:
df["онлайн"] = df["формат"] == "онлайн"
X2 = df[["посещаемость", "сдано_дз", "онлайн"]]
X2_train, X2_test, y2_train, y2_test, g2_train, g2_test = train_test_split(
    X2, y, groups, test_size=0.3, random_state=42, stratify=y
)
model2 = DecisionTreeClassifier(max_depth=3, random_state=0)
model2.fit(X2_train, y2_train)
pred2 = model2.predict(X2_test)

for group in ["очно", "онлайн"]:
    mask = (g2_test == group).to_numpy()
    print(f"{group:7}: accuracy {accuracy_score(y2_test[mask], pred2[mask]):.2f}, "
          f"recall {recall_score(y2_test[mask], pred2[mask]):.2f}")

Для онлайн-учеников стало намного лучше. Но это **не универсальный рецепт**:

- нельзя использовать **чувствительные признаки** — пол, национальность, религию — чтобы модель решала по ним о людях. Это незаконно и несправедливо;
- иногда правильнее **убрать** признак, который по-разному измеряется для групп;
- часто нужно **собрать больше данных** о малой группе;
- иногда модель для такой задачи **вообще не стоит применять** без проверки человеком.

Главный навык этого урока: **всегда считать метрики по группам**. Без этого проблему просто не увидеть.

## 5. Приватность и персональные данные

**Персональные данные** — сведения, по которым можно узнать человека: ФИО, фото, телефон, адрес, номер школы и класса вместе с именем, голос. В России их обработку регулирует закон о персональных данных.

Правила для ученика-инженера:

1. Собираешь данные о людях — спроси **согласие** и объясни, зачем.
2. Бери только то, что **нужно** для задачи.
3. **Обезличивай**: вместо ФИО — номера (`ученик_01`), без фото лиц, если они не нужны.
4. Не выкладывай таблицы с личными данными в открытый доступ и **не отправляй их ИИ-помощнику**.
5. Удаляй данные, когда проект закончен.

## 6. Ответственность: человек в контуре

Модель не может отвечать за решение — отвечают люди, которые её сделали и используют. Поэтому в серьёзных задачах:

- модель **советует**, а решает **человек**;
- у человека есть право **узнать причину** решения и **оспорить** его;
- модель регулярно **перепроверяют** на новых данных;
- заранее записывают, где модель применять **нельзя**.

Для этого инженеры заполняют **карточку ограничений** модели.

| Пункт | Пример для модели отбора на курс |
|-------|----------------------------------|
| Задача и пользователь | прогноз успеха на продвинутом курсе; использует куратор |
| Данные | 500 учеников одного кружка, 12% — онлайн |
| Качество | общая accuracy …; **по группам**: очно …, онлайн … |
| Кому может навредить | онлайн-ученикам: модель недооценивает их |
| Где нельзя применять | для окончательного отказа без разговора с учеником; в других кружках без проверки |
| Кто проверяет решения | куратор, особенно отказы |
| Как часто пересматривать | каждый учебный год на новых данных |

## 7. Практика

### Практика 1. Разбор ситуаций (10 минут)

Для каждой ситуации определи главный риск (справедливость, приватность, безопасность, прозрачность) и предложи одну меру, которая его снижает.

1. Чат-бот уверенно посоветовал дозировку лекарства, которой нет в инструкции.
2. Учебный проект выложил в открытую папку таблицу с ФИО и оценками одноклассников.
3. Модель отбора на олимпиаду обучена на данных одной сильной школы, а применяется во всём городе.
4. Банк отказал в кредите по решению модели и не объясняет причину.

| № | Главный риск | Мера |
|--:|--------------|------|
| 1 | | |
| 2 | | |
| 3 | | |
| 4 | | |

<details>
<summary>Возможные ответы</summary>

1. Безопасность: ИИ выдумал факт (галлюцинация). Мера: медицинские вопросы — только к специалисту; бот должен отказываться давать дозировки.
2. Приватность: персональные данные в открытом доступе. Мера: обезличить данные, закрыть доступ, спросить согласие.
3. Справедливость: нерепрезентативные данные. Мера: проверить метрики по школам, собрать данные других школ.
4. Прозрачность и ответственность: человек не может оспорить решение. Мера: объяснять причины и давать возможность пересмотра человеком.
</details>

### Практика 2. Метрики по классам (8 минут)

Проверим модель `model` (без признака формата) ещё по одному разрезу: одинаково ли она работает для учеников **8 и 9 класса**? Запусти ячейку — она добавит в таблицу класс каждого ученика.

Затем раздели данные ещё раз **с теми же параметрами**, что в разделе 4, но вместо `groups` передай `df["класс"]` — разбиение получится тем же самым. Посчитай accuracy и recall по каждому классу для предсказаний `pred`. Есть ли разница? Почему?

In [ ]:
random.seed(5)
classes = []
for i in range(len(df)):
    classes.append(random.choice([8, 9]))
df["класс"] = classes
print(df["класс"].value_counts())

# Практика 2

<details>
<summary>Решение</summary>

```python
X_tr, X_te, y_tr, y_te, c_train, c_test = train_test_split(
    X, y, df["класс"], test_size=0.3, random_state=42, stratify=y
)
for grade in [8, 9]:
    mask = (c_test == grade).to_numpy()
    print(f"{grade} класс: accuracy {accuracy_score(y_test[mask], pred[mask]):.2f}, "
          f"recall {recall_score(y_test[mask], pred[mask]):.2f}")
```
Большой разницы нет: класс мы назначили случайно, он не связан ни с признаками, ни с результатом. Небольшие различия — случайность маленьких групп. Проверка по группам нужна и тогда, когда мы **ожидаем**, что разницы нет: только так в этом можно убедиться.
</details>

### Практика 3. Карточка ограничений (7 минут)

Заполни карточку ограничений для модели из раздела 4 (модель `model` без признака формата) — с настоящими числами из эксперимента.

| Пункт | Мой ответ |
|-------|-----------|
| Задача и пользователь | |
| Данные | |
| Качество (общее и по группам) | |
| Кому может навредить | |
| Где нельзя применять | |
| Кто проверяет решения | |

## Работа с ИИ-помощником

- «Приведи ещё два реальных примера смещения в системах ИИ и объясни источник каждого.»
- «Какие персональные данные есть в такой таблице: …? Как их обезличить?»
- **Никогда** не отправляй помощнику таблицы с настоящими персональными данными — только обезличенные или выдуманные.

## Мини-глоссарий урока 24

| Термин | Смысл |
|--------|-------|
| **Смещение (bias)** | систематическая ошибка модели против части людей |
| **Признак-посредник** | признак, который косвенно указывает на группу |
| **Метрики по группам** | качество модели, посчитанное отдельно для каждой группы |
| **Чувствительный признак** | пол, национальность, религия и т. п. — нельзя использовать для решений о людях |
| **Персональные данные** | сведения, по которым можно узнать человека |
| **Обезличивание** | удаление из данных того, что указывает на конкретного человека |
| **Человек в контуре** | модель советует, решает человек |

## Итог урока

1. Модель честно повторяет то, что в данных, — в том числе несправедливость и неравные измерения.
2. Общая accuracy может скрывать плохую работу на малой группе: **считай метрики по группам**.
3. Персональные данные — только с согласием, обезличенно и минимально; решения о людях принимает человек.

## Домашнее задание (20 минут)

Выбери сервис с ИИ, которым ты пользуешься (рекомендации, переводчик, фильтр, помощник). Напиши: какие риски для пользователей ты видишь (минимум 3, из разных принципов) и как разработчики могли бы их снизить.

**Сервис:** …

**Риски и меры:** …

## Самопроверка

1. Что такое смещение модели? Назови три источника.
2. Почему удаление столбца «группа» не гарантирует справедливость?
3. Зачем считать метрики по группам?
4. Какие данные считаются персональными? Как их обезличить?
5. Что такое «человек в контуре»?

**Следующий урок (11.03.2027):** самостоятельная работа №3 — полный цикл машинного обучения.